In [12]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import time
import copy
from datetime import datetime



In [55]:
class Transformer_Binary_Classifier(torch.nn.Module):
        def __init__(self, input_size, hidden_size, dropout, nhead=8, num_layers=6):
            super(Transformer_Binary_Classifier, self).__init__()
            self.input_size = input_size
            self.hidden_size  = hidden_size
            self.fc1 = torch.nn.Linear(1, self.hidden_size)
            self.fc2 = torch.nn.Linear(self.hidden_size, 1)
            encoder_layer = torch.nn.TransformerEncoderLayer(d_model=hidden_size, nhead=nhead, dropout=dropout,
                                                       dim_feedforward=hidden_size)
            self.transf = torch.nn.TransformerEncoder(encoder_layer, num_layers)
            self.relu = torch.nn.ReLU()
            self.fc_out = torch.nn.Linear(input_size, 1)
            self.sigmoid = torch.nn.Sigmoid()
            
        def forward(self, x):
            x = x.unsqueeze(2)
            x = self.fc1(x)
            x = self.relu(x)
            x = self.transf(x)
            x = self.fc2(x)
            x = x.squeeze(2)
            x = self.relu(x)
            output = self.fc_out(x)
            
            output = self.sigmoid(output)
            return output

In [3]:
data = pd.read_csv('/Users/nkerstingadxnet.com/Documents/Higgs/orig/atlas-higgs-challenge-2014-v2.csv')
data['Binary_Label'] = data['Label'].map({'s':1,'b':0})
nonphi_columns = [c for c in data.columns if "phi" in c]
for col in nonphi_columns:
    data.drop(col, axis=1, inplace=True)

In [4]:
nulled_data = data.replace(-999.0, 0)
train_data = nulled_data.loc[nulled_data['KaggleSet'] == 't'].copy(deep=True)
public_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'b'].copy(deep=True)
private_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'v'].copy(deep=True)

In [5]:
orig_train_data = train_data.copy(deep=True)
orig_public_test_data = public_test_data.copy(deep=True)
orig_private_test_data = private_test_data.copy(deep=True)


# now replace the null values with column averages
train_data.drop('EventId', axis=1, inplace=True)
train_data.drop('Label', axis=1, inplace=True)
train_data.drop('Binary_Label', axis=1, inplace=True)
train_data.drop('Weight', axis=1, inplace=True)
train_data.drop('KaggleWeight', axis=1, inplace=True)
train_data.drop('KaggleSet', axis=1, inplace=True)
for col in train_data:
    train_data[col].fillna(train_data[col].mean(), inplace=True)
#train_data.isnull().sum()


public_test_data.drop('EventId', axis=1, inplace=True)
public_test_data.drop('Label', axis=1, inplace=True)
public_test_data.drop('Binary_Label', axis=1, inplace=True)
public_test_data.drop('Weight', axis=1, inplace=True)
public_test_data.drop('KaggleWeight', axis=1, inplace=True)
public_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in public_test_data:
    public_test_data[col].fillna(public_test_data[col].mean(), inplace=True)
#public_test_data.isnull().sum()


private_test_data.drop('EventId', axis=1, inplace=True)
private_test_data.drop('Label', axis=1, inplace=True)
private_test_data.drop('Binary_Label', axis=1, inplace=True)
private_test_data.drop('Weight', axis=1, inplace=True)
private_test_data.drop('KaggleWeight', axis=1, inplace=True)
private_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in private_test_data:
    private_test_data[col].fillna(private_test_data[col].mean(), inplace=True)
#private_test_data.isnull().sum()

In [6]:
# now let's normalize
normed_train_data = (train_data - train_data.min())/(train_data.max() - train_data.min())

normed_public_test_data = (public_test_data - public_test_data.min())/(public_test_data.max() - public_test_data.min())

normed_private_test_data = (private_test_data - private_test_data.min())/(private_test_data.max() - private_test_data.min())



In [9]:
train_input_data = []
for i in range(len(orig_train_data)):
   train_input_data.append([torch.tensor(normed_train_data.iloc[i], dtype=torch.float), torch.tensor(orig_train_data['Binary_Label'].iloc[i] , dtype=torch.float)])
valid_input_data = []
for i in range(len(orig_public_test_data)):
   valid_input_data.append([torch.tensor(normed_public_test_data.iloc[i], dtype=torch.float), torch.tensor(orig_public_test_data['Binary_Label'].iloc[i] , dtype=torch.float)])


train_dataloader = DataLoader(train_input_data, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_input_data, batch_size=128, shuffle=True)

In [59]:
model = Transformer_Binary_Classifier(24, 256, 0.05)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [61]:
import time
import copy
from datetime import datetime

now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')

start_time = time.time()
epoch = 3
maxvalcount = 5
minval_loss = np.inf
valcount = maxvalcount
best_model = model
for epoch in range(epoch):
    print("EPOCH: ", epoch)
    model.train()
    for i,batch in enumerate(train_dataloader):
        inputs, output = batch
        optimizer.zero_grad()
        # Forward pass
        y_pred = model(inputs)
        # Compute Loss
        loss = criterion(y_pred.squeeze(), output)
        if i % 100 == 0:
            logstring = 'Batch {}: train loss: {}'.format(i, loss.item())
            print(logstring)
            logfile.write(logstring + '\n')
        # Backward pass
        loss.backward()
        optimizer.step()
    # compute validation loss
    model.eval()
    with torch.set_grad_enabled(False):
        val_loss = 0
        for i,batch in enumerate(valid_dataloader):
            inputs, output = batch
            y_pred = model(inputs)
            y_pred = model(inputs)
            loss = criterion(y_pred.squeeze(), output)
            val_loss += loss
            if i % 100 == 0:
                logstring = 'Validation Batch {}: loss: {}'.format(i, loss.item())
                print(logstring)
                logfile.write(logstring + '\n')
        avg_val_loss = val_loss / len(valid_dataloader)
        valstring = f"AVERAGE BATCH VAL LOSS = {avg_val_loss}"
        print(valstring)
        logfile.write(valstring + '\n')
    if avg_val_loss < minval_loss:
        minval_loss = avg_val_loss
        best_model = copy.deepcopy(model)
        valcount = maxvalcount
    else:
        valcount -= 1
    if valcount == 0:
        endmsg = f"Validation Loss failed to decrease in {maxvalcount} epochs, exiting with best model, val loss = {minval_loss}"
        print(endmsg)
        logfile.write(endmsg + '\n')
        break

end_time = time.time()
timestring = f"Time elapsed in training: {end_time - start_time} seconds"
print(timestring)
logfile.write(timestring + '\n')

EPOCH:  0
Batch 0: train loss: 0.4571743309497833
Batch 100: train loss: 0.5739582777023315
Batch 200: train loss: 0.5193415880203247
Batch 300: train loss: 0.45577099919319153
Batch 400: train loss: 0.4097275733947754
Batch 500: train loss: 0.4962128698825836
Batch 600: train loss: 0.48543405532836914
Batch 700: train loss: 0.41680070757865906
Batch 800: train loss: 0.5423620343208313
Batch 900: train loss: 0.5253950953483582
Batch 1000: train loss: 0.4274865686893463
Batch 1100: train loss: 0.5211706161499023
Batch 1200: train loss: 0.4756510853767395
Batch 1300: train loss: 0.5202255249023438
Batch 1400: train loss: 0.5365517735481262
Batch 1500: train loss: 0.4287196397781372
Batch 1600: train loss: 0.43404555320739746
Batch 1700: train loss: 0.5385883450508118
Batch 1800: train loss: 0.421478807926178
Batch 1900: train loss: 0.3783802390098572
Validation Batch 0: loss: 0.5610747337341309
Validation Batch 100: loss: 0.5611539483070374
Validation Batch 200: loss: 0.5368117094039917


53

In [31]:
x = torch.ones(24)

In [32]:
x.shape

torch.Size([24])

In [37]:
y = x.unsqueeze(1)

In [38]:
y.shape

torch.Size([24, 1])

In [39]:
z = torch.rand(1,32)

In [41]:
w = torch.matmul(y,z)

In [42]:
w.shape

torch.Size([24, 32])

In [43]:
t = torch.rand(32,1)

In [44]:
a = torch.matmul(w,t)
a.shape

torch.Size([24, 1])

In [48]:
torch.matmul(a.squeeze(1), torch.rand(24,1)).shape

torch.Size([1])

In [ ]:
torch.matmul(a.squeeze(1), torch.rand(24,1)).shape

In [53]:
x = torch.rand(128,24)
y = x.unsqueeze(2)
y.shape

torch.Size([128, 24, 1])

In [54]:
y.squeeze(2).shape

torch.Size([128, 24])